# 第01章：Hello Triton — 你的第一个 GPU Kernel

## 前置知识
- Python 基础
- PyTorch 张量基本操作
- 了解 GPU 并行计算的概念（不需要会 CUDA）

## 学习目标
- 理解 Triton 是什么，为什么需要它
- 编写并运行第一个 Triton kernel（向量加法）
- 掌握 `@triton.jit`、`tl.program_id`、`tl.load`、`tl.store` 的基本用法
- 理解 Triton 与 CUDA 编程模型的核心区别

## 1.1 什么是 Triton？

**Triton** 是由 OpenAI 开发的 GPU 编程语言和编译器。它的设计目标是：

```
编程难度      性能
  ↑            ↑
  |  CUDA ●----●  CUDA        ← 最强性能，但编程极其复杂
  |            |
  |  Triton ●--●  Triton      ← 接近 CUDA 性能，编程简单得多
  |            |
  |  PyTorch ●-●  PyTorch     ← 最简单，但性能受限
  ↓            ↓
```

### Triton vs CUDA 核心区别

| 特性 | CUDA | Triton |
|------|------|--------|
| 编程粒度 | **线程级** (每个线程处理1个元素) | **块级** (每个 program 处理一个数据块) |
| 共享内存 | 手动管理 `__shared__` | 编译器自动管理 |
| 内存合并 | 手动优化访问模式 | 编译器自动优化 |
| 同步 | 手动 `__syncthreads()` | 隐式同步 |
| 向量化 | 手动 `float4` 等 | 编译器自动向量化 |
| Tensor Core | 手动调用 WMMA/MMA | `tl.dot()` 自动映射 |

**一句话总结**：Triton 把 CUDA 中最痛苦的底层优化交给编译器，你只需要描述算法逻辑。

## 1.2 Triton 编程模型

在 CUDA 中，你为**每个线程**写代码：
```
CUDA: 1 thread → 1 element
      Grid → Block → Thread
```

在 Triton 中，你为**每个 program（程序实例）**写代码，每个 program 处理**一块数据**：
```
Triton: 1 program → BLOCK_SIZE elements
        Grid → Program
```

### 执行模型示意图

假设我们有 8 个元素，BLOCK_SIZE=4，需要 2 个 program：

```
数据:  [a0, a1, a2, a3, a4, a5, a6, a7]
        |_____________|  |_____________|
        Program 0        Program 1
        (pid=0)          (pid=1)
        处理 4 个元素     处理 4 个元素
```

每个 program 内部，Triton 编译器会自动将工作分配到 GPU 的线程上。你不需要关心线程。

In [ ]:
# 首先，导入必要的库
import torch
import triton
import triton.language as tl

print(f"PyTorch 版本: {torch.__version__}")
print(f"Triton 版本: {triton.__version__}")
print(f"CUDA 是否可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 设备: {torch.cuda.get_device_name(0)}")

## 1.3 第一个 Triton Kernel：向量加法

我们来实现最简单的操作：**C = A + B**（逐元素加法）

### 算法思路
```
A:  [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]
B:  [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
                      ↓ 逐元素相加
C:  [1.1, 2.2, 3.3, 4.4, 5.5, 6.6, 7.7, 8.8]
```

分块处理（假设 BLOCK_SIZE=4）：
```
Program 0 (pid=0):  处理 A[0:4] + B[0:4] → C[0:4]
Program 1 (pid=1):  处理 A[4:8] + B[4:8] → C[4:8]
```

In [ ]:
@triton.jit  # 这个装饰器告诉 Triton：这是一个 GPU kernel 函数
def vector_add_kernel(
    # --- 指针参数：指向 GPU 内存中的数据 ---
    a_ptr,      # 输入向量 A 的指针
    b_ptr,      # 输入向量 B 的指针
    c_ptr,      # 输出向量 C 的指针
    # --- 标量参数 ---
    n_elements,  # 向量长度
    # --- 编译时常量 ---
    BLOCK_SIZE: tl.constexpr,  # 每个 program 处理的元素数量
):
    """
    向量加法 kernel: C = A + B
    
    每个 program instance 处理 BLOCK_SIZE 个元素。
    """
    # ============================================================
    # 第1步：确定当前 program 处理哪些元素
    # ============================================================
    # tl.program_id(0) 返回当前 program 在第 0 维的索引
    # 类似 CUDA 的 blockIdx.x
    pid = tl.program_id(axis=0)
    
    # 计算当前 program 负责的元素偏移量
    # 例如: pid=0 → offsets=[0,1,2,3], pid=1 → offsets=[4,5,6,7]
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    #                       ^^^^^^^^^^^^^^^^^^^^^^^^
    #                       生成 [0, 1, 2, ..., BLOCK_SIZE-1]
    
    # ============================================================
    # 第2步：创建边界 mask（防止越界访问）
    # ============================================================
    # 当 n_elements 不是 BLOCK_SIZE 的整数倍时，最后一个 program
    # 的部分 offset 会超出数组范围，需要 mask 来保护
    mask = offsets < n_elements
    
    # ============================================================
    # 第3步：从全局内存加载数据
    # ============================================================
    # tl.load: 从 GPU 全局内存加载一个块的数据
    # - a_ptr + offsets: 指针 + 偏移 = 实际的内存地址
    # - mask: True 的位置才会真正加载
    # - other: mask 为 False 的位置填充的默认值
    a = tl.load(a_ptr + offsets, mask=mask, other=0.0)
    b = tl.load(b_ptr + offsets, mask=mask, other=0.0)
    
    # ============================================================
    # 第4步：计算（这就是全部的算法逻辑！）
    # ============================================================
    c = a + b
    
    # ============================================================
    # 第5步：将结果写回全局内存
    # ============================================================
    tl.store(c_ptr + offsets, c, mask=mask)

### 代码解析

让我们逐行理解上面的 kernel：

```
① @triton.jit          → 标记为 GPU kernel
② tl.program_id(0)     → 获取当前 program 的 ID（类似 CUDA blockIdx.x）
③ tl.arange(0, N)      → 生成连续整数序列 [0, 1, ..., N-1]
④ mask = offsets < n    → 边界检查（布尔向量）
⑤ tl.load(ptr, mask)   → 从 GPU 内存读取一块数据
⑥ c = a + b            → 块级运算（整个块同时计算）
⑦ tl.store(ptr, c, mask) → 将结果写回 GPU 内存
```

**关键理解**：在 Triton 中，`a`、`b`、`c` 不是单个数字，而是**长度为 BLOCK_SIZE 的向量**！所有操作都是块级的。

In [ ]:
def vector_add(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """
    Triton kernel 的 Python 包装函数。
    
    Triton kernel 本身只是描述每个 program 的计算逻辑，
    这个包装函数负责：
    1. 分配输出张量
    2. 计算 grid（需要多少个 program）
    3. 启动 kernel
    """
    # 检查输入
    assert a.is_cuda and b.is_cuda, "输入必须在 GPU 上！"
    assert a.shape == b.shape, "形状必须相同！"
    
    # 分配输出
    c = torch.empty_like(a)
    n_elements = a.numel()
    
    # 设置 BLOCK_SIZE 和 grid
    BLOCK_SIZE = 1024  # 每个 program 处理 1024 个元素
    
    # grid: 需要多少个 program 实例
    # 向上取整确保覆盖所有元素
    grid = (triton.cdiv(n_elements, BLOCK_SIZE),)  # cdiv = ceiling division
    # 例如: n=8000, BLOCK_SIZE=1024 → grid = (8,)
    # 意味着启动 8 个 program 实例
    
    # 启动 kernel!
    vector_add_kernel[grid](
        a, b, c,           # 指针参数（PyTorch tensor 自动转为指针）
        n_elements,        # 标量参数
        BLOCK_SIZE=BLOCK_SIZE,  # 编译时常量
    )
    
    return c

In [ ]:
# 测试！
torch.manual_seed(42)
n = 98432  # 故意选一个不是 1024 整数倍的数，测试边界处理

a = torch.randn(n, device='cuda', dtype=torch.float32)
b = torch.randn(n, device='cuda', dtype=torch.float32)

# Triton 版本
c_triton = vector_add(a, b)

# PyTorch 版本（参考答案）
c_torch = a + b

# 验证正确性
print(f"输入大小: {n}")
print(f"最大误差: {(c_triton - c_torch).abs().max().item():.2e}")
print(f"结果是否一致: {torch.allclose(c_triton, c_torch)}")
print(f"\n前 5 个结果:")
print(f"  Triton:  {c_triton[:5].tolist()}")
print(f"  PyTorch: {c_torch[:5].tolist()}")

## 1.4 Kernel 启动过程详解

```
Python 代码                          GPU
===========                          ===

vector_add_kernel[grid](...)    →    启动 grid[0] 个 program 实例
                                     ┌──────────────────────────┐
grid = (8,)                          │  Program 0  (pid=0)     │
表示启动 8 个 program                 │  Program 1  (pid=1)     │
                                     │  Program 2  (pid=2)     │
BLOCK_SIZE = 1024                    │  ...                    │
每个 program 处理                     │  Program 7  (pid=7)     │
1024 个元素                          └──────────────────────────┘
                                     每个 program 并行执行！
```

### kernel[grid](...) 的语法

```python
# 方式1: grid 是一个元组
kernel[(8,)](args...)

# 方式2: grid 是一个返回元组的函数（更灵活）
grid = lambda meta: (triton.cdiv(n, meta['BLOCK_SIZE']),)
kernel[grid](args..., BLOCK_SIZE=1024)
```

## 1.5 性能对比

让我们比较 Triton 和 PyTorch 的性能：

In [ ]:
# 性能基准测试
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=['size'],  # x 轴变量名
        x_vals=[2**i for i in range(12, 25)],  # x 轴的值: 4K → 16M
        x_log=True,
        line_arg='provider',
        line_vals=['triton', 'torch'],
        line_names=['Triton', 'PyTorch'],
        styles=[('blue', '-'), ('green', '-')],
        ylabel='GB/s',
        plot_name='向量加法性能对比',
        args={},
    )
)
def benchmark(size, provider):
    a = torch.randn(size, device='cuda', dtype=torch.float32)
    b = torch.randn(size, device='cuda', dtype=torch.float32)
    quantiles = [0.5, 0.2, 0.8]  # 中位数, 20%, 80% 分位数
    if provider == 'triton':
        ms, min_ms, max_ms = triton.testing.do_bench(lambda: vector_add(a, b), quantiles=quantiles)
    else:
        ms, min_ms, max_ms = triton.testing.do_bench(lambda: a + b, quantiles=quantiles)
    # 计算带宽: 读 2 个向量 + 写 1 个向量 = 3 * size * 4 bytes
    gbps = lambda ms: 3 * size * a.element_size() / ms * 1e-6
    return gbps(ms), gbps(max_ms), gbps(min_ms)

benchmark.run(print_data=True, show_plots=True)

## 1.6 与 CUDA 的对比

同样的向量加法，CUDA 需要这样写：

```c
// CUDA 版本 —— 需要手动管理线程
__global__ void vector_add_cuda(float* A, float* B, float* C, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;  // 计算全局线程 ID
    if (idx < n) {                                     // 边界检查
        C[idx] = A[idx] + B[idx];                      // 每个线程处理 1 个元素
    }
}

// 启动: vector_add_cuda<<<ceil(n/256), 256>>>(A, B, C, n);
```

**对比**：
- CUDA：每个线程处理 1 个元素，你需要计算全局索引
- Triton：每个 program 处理 BLOCK_SIZE 个元素，用 `tl.arange` 生成偏移量

对于向量加法这样的简单操作差别不大，但对于矩阵乘法等复杂操作，Triton 的优势是巨大的。

## 1.7 总结

本章你学到了：

| 概念 | 说明 |
|------|------|
| `@triton.jit` | 声明 Triton kernel |
| `tl.program_id(axis)` | 获取当前 program 的 ID |
| `tl.arange(start, end)` | 生成整数序列 |
| `tl.load(ptr, mask)` | 从 GPU 内存加载数据 |
| `tl.store(ptr, val, mask)` | 将数据写入 GPU 内存 |
| `tl.constexpr` | 编译时常量参数 |
| `kernel[grid](args)` | 启动 kernel |
| `triton.cdiv(a, b)` | 向上取整除法 |

## 练习

1. **修改 BLOCK_SIZE**：试试 256, 512, 2048，观察性能变化
2. **实现向量减法**：修改 kernel 计算 C = A - B
3. **实现向量乘法**：修改 kernel 计算 C = A * B
4. **实现 SAXPY**：实现 C = alpha * A + B，其中 alpha 是标量
5. **思考**：为什么 BLOCK_SIZE 必须是 `tl.constexpr`？（提示：编译器需要在编译时知道块大小）

In [ ]:
# 练习4 参考答案: SAXPY kernel
@triton.jit
def saxpy_kernel(a_ptr, b_ptr, c_ptr, alpha, n_elements, BLOCK_SIZE: tl.constexpr):
    """C = alpha * A + B"""
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    a = tl.load(a_ptr + offsets, mask=mask)
    b = tl.load(b_ptr + offsets, mask=mask)
    c = alpha * a + b
    tl.store(c_ptr + offsets, c, mask=mask)

# 测试 SAXPY
n = 10000
alpha = 2.5
a = torch.randn(n, device='cuda')
b = torch.randn(n, device='cuda')
c = torch.empty_like(a)

grid = (triton.cdiv(n, 1024),)
saxpy_kernel[grid](a, b, c, alpha, n, BLOCK_SIZE=1024)

expected = alpha * a + b
print(f"SAXPY 正确性: {torch.allclose(c, expected)}")